# CryptoScope: Section 1 - Data Collection

**Project:** Crypto Market Intelligence Analyzer  
**Author:** Ganesh  
**Objective:** Collect cryptocurrency market data from multiple sources using API calls and HTML web scraping, then merge into a single master dataset ready for analysis.

---

## Data Sources

| Source | Method | Data Collected |
|--------|--------|----------------|
| CoinGecko API | REST API (JSON) | 1000 coins - price, volume, market cap, % changes |
| Fear & Greed Index API | REST API (JSON) | 30 days of market sentiment scores |
| CoinDesk News | BeautifulSoup HTML Scraping | 100+ crypto news headlines |

---

## Step 0: Install and Import Libraries

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', 'numpy', 'beautifulsoup4', 'lxml'], 
               capture_output=True)

CompletedProcess(args=['pip', 'install', 'requests', 'pandas', 'numpy', 'beautifulsoup4', 'lxml'], returncode=0, stdout=b'Requirement already satisfied: requests in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (2.32.5)\r\nRequirement already satisfied: pandas in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (3.0.2)\r\nRequirement already satisfied: numpy in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (2.4.4)\r\nRequirement already satisfied: beautifulsoup4 in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (4.14.3)\r\nRequirement already satisfied: lxml in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (6.0.2)\r\nRequirement already satisfied: charset_normalizer<4,>=2 in C:\\Users\\deshm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages (from requests) (3.4.5)\r\nRequirement already satisfied: 

In [2]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')
print(f'Run date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

Libraries imported successfully
Run date: 2026-04-27 14:39


---

## Part 1: CoinGecko API - 1000 Cryptocurrency Coins

**Method:** REST API (JSON)  
**Reason:** CoinGecko's website is JavaScript-rendered (dynamic content), which requires Selenium to scrape HTML. Their free public API returns the same structured data in JSON format, which is more reliable and reproducible for a 1000-record dataset.  
**Plan:** 4 API calls × 250 coins per page = 1000 coins total

In [3]:
def fetch_with_retry(url, params=None, retries=3, delay=10, headers=None):
    """
    Fetch a URL with exponential backoff retry logic.
    Handles rate limiting (429), timeouts, and connection errors gracefully.
    Returns parsed JSON on success, None on failure.
    """
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=15)
            
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait_time = delay * attempt
                print(f'  Rate limited (429). Waiting {wait_time}s - attempt {attempt}/{retries}')
                time.sleep(wait_time)
            else:
                print(f'  HTTP {resp.status_code} on attempt {attempt}/{retries}')
                time.sleep(delay)
                
        except requests.exceptions.Timeout:
            print(f'  Timeout on attempt {attempt}/{retries}')
            time.sleep(delay)
        except requests.exceptions.ConnectionError:
            print(f'  Connection error on attempt {attempt}/{retries}')
            time.sleep(delay * attempt)
        except Exception as e:
            print(f'  Unexpected error on attempt {attempt}: {e}')
            time.sleep(delay)
    
    print(f'  All {retries} retries failed.')
    return None


print('Retry utility function defined.')

Retry utility function defined.


In [4]:
BASE_URL = 'https://api.coingecko.com/api/v3/coins/markets'
all_coins = []

print('Fetching 1000 coins from CoinGecko...')
print('-' * 40)

for page in range(1, 5):
    print(f'Page {page}/4 ...', end=' ')
    
    params = {
        'vs_currency': 'usd',
        'order': 'market_cap_desc',
        'per_page': 250,
        'page': page,
        'sparkline': False,
        'price_change_percentage': '24h,7d,30d'
    }
    
    data = fetch_with_retry(BASE_URL, params=params)
    
    if data:
        all_coins.extend(data)
        print(f'Got {len(data)} coins. Running total: {len(all_coins)}')
    else:
        print(f'Page {page} failed - skipping.')
    
    time.sleep(3)

print('-' * 40)
print(f'Total coins fetched: {len(all_coins)}')

Fetching 1000 coins from CoinGecko...
----------------------------------------
Page 1/4 ... Got 250 coins. Running total: 250
Page 2/4 ... Got 250 coins. Running total: 500
Page 3/4 ... Got 250 coins. Running total: 750
Page 4/4 ... Got 250 coins. Running total: 1000
----------------------------------------
Total coins fetched: 1000


In [5]:
df_crypto = pd.DataFrame(all_coins)

columns_needed = [
    'id', 'symbol', 'name',
    'current_price', 'market_cap', 'market_cap_rank',
    'total_volume', 'high_24h', 'low_24h',
    'price_change_24h',
    'price_change_percentage_24h',
    'price_change_percentage_7d_in_currency',
    'price_change_percentage_30d_in_currency',
    'circulating_supply', 'total_supply',
    'ath', 'ath_change_percentage', 'atl',
    'last_updated'
]

df_crypto = df_crypto[columns_needed]

df_crypto.columns = [
    'coin_id', 'symbol', 'name',
    'price_usd', 'market_cap_usd', 'market_cap_rank',
    'volume_24h_usd', 'high_24h_usd', 'low_24h_usd',
    'price_change_24h_usd',
    'price_change_pct_24h',
    'price_change_pct_7d',
    'price_change_pct_30d',
    'circulating_supply', 'total_supply',
    'all_time_high_usd', 'ath_change_percentage', 'all_time_low_usd',
    'last_updated'
]

df_crypto['volatility_24h_usd'] = df_crypto['high_24h_usd'] - df_crypto['low_24h_usd']
df_crypto['volatility_24h_pct'] = (df_crypto['volatility_24h_usd'] / df_crypto['low_24h_usd'].replace(0, np.nan)) * 100
df_crypto['scraped_at'] = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

print(f'CoinGecko data structured: {df_crypto.shape}')
print(f'Columns: {list(df_crypto.columns)}')
df_crypto.head(3)

CoinGecko data structured: (1000, 22)
Columns: ['coin_id', 'symbol', 'name', 'price_usd', 'market_cap_usd', 'market_cap_rank', 'volume_24h_usd', 'high_24h_usd', 'low_24h_usd', 'price_change_24h_usd', 'price_change_pct_24h', 'price_change_pct_7d', 'price_change_pct_30d', 'circulating_supply', 'total_supply', 'all_time_high_usd', 'ath_change_percentage', 'all_time_low_usd', 'last_updated', 'volatility_24h_usd', 'volatility_24h_pct', 'scraped_at']


,coin_id,symbol,name,price_usd,market_cap_usd,market_cap_rank,volume_24h_usd,high_24h_usd,low_24h_usd,price_change_24h_usd,...,price_change_pct_30d,circulating_supply,total_supply,all_time_high_usd,ath_change_percentage,all_time_low_usd,last_updated,volatility_24h_usd,volatility_24h_pct,scraped_at
0,bitcoin,btc,Bitcoin,77834.00,1557902791627,1,3.260366e+10,79400.00,77595.000000,-158.649487,...,17.015695,2.002136e+07,2.002136e+07,126080.00,-38.26574,67.810000,2026-04-27T09:09:57.434Z,1805.000000,2.326181,2026-04-27 09:10:13
1,ethereum,eth,Ethereum,2320.39,279914034732,2,1.440008e+10,2398.26,2311.600000,-10.314777,...,15.679054,1.206886e+08,1.206886e+08,4946.05,-53.08604,0.432979,2026-04-27T09:09:58.153Z,86.660000,3.748918,2026-04-27 09:10:13
2,tether,usdt,Tether,1.00,189765201459,3,5.379215e+10,1.00,0.999996,-0.000112,...,0.069742,1.897646e+11,1.952316e+11,1.32,-24.41806,0.572521,2026-04-27T09:09:57.644Z,0.000004,0.000400,2026-04-27 09:10:13


---

## Part 2: Fear & Greed Index API - Market Sentiment

**What it measures:** Overall crypto market emotion on a 0-100 scale.  
**Why it matters:** Sentiment is a proven leading indicator for crypto price movements.  
**Engineering:** We extract three non-constant features from the 30-day history:
- `fear_greed_score` - today's score
- `fear_greed_7d_avg` - 7-day rolling average (trend)
- `sentiment_shift` - difference between today and 7 days ago (momentum)

In [ ]:
FG_URL = 'https://api.alternative.me/fng/?limit=30'

print('Fetching Fear & Greed Index (30 days)...')
fg_response = fetch_with_retry(FG_URL)

if fg_response:
    df_fg = pd.DataFrame(fg_response['data'])
    df_fg = df_fg[['value', 'value_classification', 'timestamp']]
    df_fg.columns = ['fear_greed_score', 'fear_greed_label', 'fg_timestamp']
    df_fg['fear_greed_score'] = df_fg['fear_greed_score'].astype(int)
    df_fg['fg_date'] = pd.to_datetime(df_fg['fg_timestamp'].astype(int), unit='s')
    df_fg = df_fg.sort_values('fg_date', ascending=False).reset_index(drop=True)

    today_score = int(df_fg.iloc[0]['fear_greed_score'])
    today_label = df_fg.iloc[0]['fear_greed_label']
    avg_7d = round(df_fg.head(7)['fear_greed_score'].mean(), 1)
    score_7d_ago = int(df_fg.iloc[6]['fear_greed_score'])
    sentiment_shift = today_score - score_7d_ago

    print(f'Fear & Greed data fetched: {len(df_fg)} days')
    print(f'  Today score   : {today_score} ({today_label})')
    print(f'  7-day average : {avg_7d}')
    print(f'  Sentiment shift (vs 7d ago): {sentiment_shift:+d}')
    
    import os
    os.makedirs('../data', exist_ok=True)
    df_fg.to_csv('../data/fear_greed.csv', index=False)
    print('fear_greed.csv saved.')
    
else:
    print('Fear & Greed fetch failed. Using neutral defaults.')
    today_score = 50
    today_label = 'Neutral'
    avg_7d = 50
    sentiment_shift = 0

print(df_fg.head())

Fetching Fear & Greed Index (30 days)...
Fear & Greed data fetched: 30 days
  Today score   : 47 (Neutral)
  7-day average : 37.3
  Sentiment shift (vs 7d ago): +14
fear_greed.csv saved.
   fear_greed_score fear_greed_label fg_timestamp    fg_date
0                47          Neutral   1777248000 2026-04-27
1                33             Fear   1777161600 2026-04-26
2                31             Fear   1777075200 2026-04-25
3                39             Fear   1776988800 2026-04-24
4                46             Fear   1776902400 2026-04-23


---

## Part 3: CoinDesk News - HTML Scraping with BeautifulSoup

**Method:** `requests` + `BeautifulSoup` (HTML parsing)  
**Why this qualifies as web scraping:** CoinDesk has no free public API. We fetch raw HTML pages and parse the DOM structure to extract article titles, dates, and URLs - this is standard web scraping.  
**Target:** 10 pages of market news headlines  
**Output:** Headline text matched to coins by symbol/name mention count

In [7]:
BROWSER_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

articles = []
scrape_date = datetime.utcnow().strftime('%Y-%m-%d')

urls_to_scrape = [
    'https://coinpaprika.com/news/',
    'https://coinpaprika.com/news/?page=2',
    'https://coinpaprika.com/news/?page=3',
    'https://coinpaprika.com/news/?page=4',
    'https://coinpaprika.com/news/?page=5',
    'https://coinpaprika.com/news/?page=6',
    'https://coinpaprika.com/news/?page=7',
    'https://coinpaprika.com/news/?page=8',
    'https://coinpaprika.com/news/?page=9',
    'https://coinpaprika.com/news/?page=10',
]

print('Scraping crypto news headlines using BeautifulSoup...')
print('-' * 50)

for i, url in enumerate(urls_to_scrape, 1):
    try:
        resp = requests.get(url, headers=BROWSER_HEADERS, timeout=15)
        print(f'Page {i:2d} | Status: {resp.status_code}', end=' | ')

        if resp.status_code != 200:
            print('Skipped')
            time.sleep(5)
            continue

        soup = BeautifulSoup(resp.text, 'lxml')

        page_count = 0
        
        for tag in soup.find_all(['h1', 'h2', 'h3', 'h4']):
            text = tag.get_text(strip=True)
            parent_a = tag.find('a') or tag.find_parent('a')
            href = parent_a['href'] if parent_a and parent_a.get('href') else ''
            
            if len(text) > 25 and len(text) < 300:
                articles.append({
                    'headline': text,
                    'url': href if href.startswith('http') else 'https://coinpaprika.com' + href,
                    'source': 'coinpaprika',
                    'scraped_page': i,
                    'scraped_date': scrape_date
                })
                page_count += 1
        
        print(f'Headlines found: {page_count}')
        time.sleep(2)

    except requests.exceptions.Timeout:
        print('Timeout - skipped')
        time.sleep(5)
    except Exception as e:
        print(f'Error: {e}')
        time.sleep(5)

df_news = pd.DataFrame(articles).drop_duplicates(subset=['headline']).reset_index(drop=True)
print('-' * 50)
print(f'Total unique headlines scraped: {len(df_news)}')

Scraping crypto news headlines using BeautifulSoup...
--------------------------------------------------
Page  1 | Status: 200 | Headlines found: 1
Page  2 | Status: 200 | Headlines found: 1
Page  3 | Status: 200 | Headlines found: 1
Page  4 | Status: 200 | Headlines found: 1
Page  5 | Status: 200 | Headlines found: 1
Page  6 | Status: 200 | Headlines found: 1
Page  7 | Status: 200 | Headlines found: 1
Page  8 | Status: 200 | Headlines found: 1
Page  9 | Status: 200 | Headlines found: 1
Page 10 | Status: 200 | Headlines found: 1
--------------------------------------------------
Total unique headlines scraped: 1


In [ ]:
print('Matching headlines to coins by symbol and name...')

headlines_text = ' '.join(df_news['headline'].str.lower().tolist())

def count_mentions(symbol, name, all_headlines):
    sym_lower = symbol.lower()
    name_lower = name.lower()
    count = 0
    for h in all_headlines:
        h_lower = h.lower()
        if sym_lower in h_lower.split() or name_lower in h_lower:
            count += 1
    return count

headlines_list = df_news['headline'].tolist()

df_crypto['news_mentions'] = df_crypto.apply(
    lambda row: count_mentions(row['symbol'], row['name'], headlines_list),
    axis=1
)

print(f'Mentions mapped. Coins with at least 1 mention: {(df_crypto["news_mentions"] > 0).sum()}')
print('\nTop 10 most mentioned coins:')
print(df_crypto[['name', 'symbol', 'news_mentions']]
      .sort_values('news_mentions', ascending=False)
      .head(10)
      .to_string(index=False))

df_news.to_csv('../data/news_data.csv', index=False)

print(f'\nnews_data.csv saved: {len(df_news)} rows')

Matching headlines to coins by symbol and name...
Mentions mapped. Coins with at least 1 mention: 0

Top 10 most mentioned coins:
        name     symbol  news_mentions
     Bitcoin        btc              0
    Ethereum        eth              0
      Tether       usdt              0
         XRP        xrp              0
         BNB        bnb              0
        USDC       usdc              0
      Solana        sol              0
        TRON        trx              0
Figure Heloc figr_heloc              0
    Dogecoin       doge              0

news_data.csv saved: 1 rows


---

## Part 4: Merge All Sources into Master Dataset

In [9]:
df_master = df_crypto.copy()

df_master['fear_greed_score'] = today_score
df_master['fear_greed_label'] = today_label
df_master['fear_greed_7d_avg'] = avg_7d
df_master['sentiment_shift'] = sentiment_shift

def get_volatility_category(pct):
    if pd.isna(pct): return 'Unknown'
    if pct < 2: return 'Low'
    elif pct < 5: return 'Medium'
    elif pct < 10: return 'High'
    else: return 'Extreme'

def get_risk_tier(mcap):
    if pd.isna(mcap): return 'Unknown'
    if mcap > 10e9: return 'Mega'
    elif mcap > 1e9: return 'Large'
    elif mcap > 100e6: return 'Mid'
    else: return 'Small'

def get_market_signal(row):
    pct_30d = row['price_change_pct_30d']
    if pd.isna(pct_30d): return 'Unknown'
    if pct_30d > 10: return 'Bullish'
    elif pct_30d > 0: return 'Neutral'
    elif pct_30d > -15: return 'Bearish'
    else: return 'Strong Bear'

df_master['volatility_category'] = df_master['volatility_24h_pct'].apply(get_volatility_category)
df_master['risk_tier'] = df_master['market_cap_usd'].apply(get_risk_tier)
df_master['market_signal'] = df_master.apply(get_market_signal, axis=1)

stablecoins = ['usdt','usdc','busd','dai','tusd','usdp','gusd','frax','lusd','usdd','pyusd','fdusd']
df_master['is_stablecoin'] = df_master['symbol'].str.lower().isin(stablecoins).astype(int)

df_master['log_price'] = np.log1p(df_master['price_usd'])
df_master['log_market_cap'] = np.log1p(df_master['market_cap_usd'])
df_master['log_volume'] = np.log1p(df_master['volume_24h_usd'])
df_master['momentum_score'] = (
    df_master['price_change_pct_24h'].fillna(0) * 0.5 +
    df_master['price_change_pct_30d'].fillna(0) * 0.5
)
df_master['pct_below_ath'] = df_master['ath_change_percentage']

print(f'Master dataset created: {df_master.shape}')
print(f'Columns ({df_master.shape[1]} total):')
for i, col in enumerate(df_master.columns, 1):
    print(f'  {i:2}. {col}')

Master dataset created: (1000, 36)
Columns (36 total):
   1. coin_id
   2. symbol
   3. name
   4. price_usd
   5. market_cap_usd
   6. market_cap_rank
   7. volume_24h_usd
   8. high_24h_usd
   9. low_24h_usd
  10. price_change_24h_usd
  11. price_change_pct_24h
  12. price_change_pct_7d
  13. price_change_pct_30d
  14. circulating_supply
  15. total_supply
  16. all_time_high_usd
  17. ath_change_percentage
  18. all_time_low_usd
  19. last_updated
  20. volatility_24h_usd
  21. volatility_24h_pct
  22. scraped_at
  23. news_mentions
  24. fear_greed_score
  25. fear_greed_label
  26. fear_greed_7d_avg
  27. sentiment_shift
  28. volatility_category
  29. risk_tier
  30. market_signal
  31. is_stablecoin
  32. log_price
  33. log_market_cap
  34. log_volume
  35. momentum_score
  36. pct_below_ath


In [ ]:
df_master.to_csv('../data/cryptoscope_master.csv', index=False)

print('=' * 55)
print('CRYPTOSCOPE - DATA COLLECTION COMPLETE')
print('=' * 55)
print(f'''
Source 1 - CoinGecko API
  Method  : REST API with retry logic
  Records : {len(df_crypto)} coins
  Fields  : price, volume, market cap, % changes, ATH

Source 2 - Fear and Greed Index API
  Method  : REST API with retry logic
  Records : 30 daily sentiment scores
  Features: today score={today_score}, 7d avg={avg_7d}, shift={sentiment_shift:+d}

Source 3 - CoinPaprika News (BeautifulSoup HTML Scraping)
  Method  : requests + BeautifulSoup HTML parsing
  Records : {len(df_news)} unique headlines
  Fields  : headline, url, source, date

Master Dataset
  Rows    : {len(df_master)}
  Columns : {df_master.shape[1]}
  File    : cryptoscope_master.csv
''')
print('=' * 55)

CRYPTOSCOPE - DATA COLLECTION COMPLETE

Source 1 - CoinGecko API
  Method  : REST API with retry logic
  Records : 1000 coins
  Fields  : price, volume, market cap, % changes, ATH

Source 2 - Fear and Greed Index API
  Method  : REST API with retry logic
  Records : 30 daily sentiment scores
  Features: today score=47, 7d avg=37.3, shift=+14

Source 3 - CoinPaprika News (BeautifulSoup HTML Scraping)
  Method  : requests + BeautifulSoup HTML parsing
  Records : 1 unique headlines
  Fields  : headline, url, source, date

Master Dataset
  Rows    : 1000
  Columns : 36
  File    : cryptoscope_master.csv

